正式月更入口

In [ ]:
from tp_data.monthly_update import build_default_paths, run_monthly_update

paths = build_default_paths()
paths


In [ ]:
# 更新模式可选: both / screen_only / returns_only
update_mode = "both"

# 每月只需要设置生产输入批次。文件放在:
# production_inputs/incoming/YYYYMM/screen
# production_inputs/incoming/YYYYMM/returns
# production_inputs/incoming/YYYYMM/ciq
input_month = "202606"

# 通常保持 None，让 run_monthly_update() 从 input_month 批次自动选择文件。
# 如需临时覆盖，可以填入绝对路径。
screen_excel = None
returns_delta = None
ciq_dir = None
fs_sector_dir = None
skip_fs_sector = False

# dry_run=True 只做读取、合并和 QA 校验，不写入 canonical parquet。
dry_run = False


In [ ]:
result = run_monthly_update(
    update_mode=update_mode,
    input_month=input_month,
    screen_excel=screen_excel,
    returns_delta=returns_delta,
    ciq_dir=ciq_dir,
    fs_sector_dir=fs_sector_dir,
    skip_fs_sector=skip_fs_sector,
    dry_run=dry_run,
)
result


In [ ]:
import pandas as pd

screen_agg = pd.read_parquet(paths["screen_path"], columns=["Date"])
last_screen = pd.read_parquet(paths["last_screen_path"], columns=["Date"])

pd.DataFrame(
    {
        "item": [
            "update_mode",
            "update_screen",
            "update_returns",
            "month_date",
            "screen_rows",
            "screen_last_date",
            "last_screen_rows",
            "last_screen_date",
            "returns_last_date",
        ],
        "value": [
            result["update_mode"],
            result["update_screen"],
            result["update_returns"],
            result["month_date"],
            len(screen_agg),
            screen_agg["Date"].max(),
            len(last_screen),
            last_screen["Date"].max(),
            result.get("returns_last_date"),
        ],
    }
)


## 缺失值审核

对最新月末与历史数据做缺失值对比。

- 全量公司
- `Weight in MSCI WORLD > 0`
- `Weight in SP500 > 0`
- `Weight in STOXX EUROPE 600 > 0`

In [ ]:
from IPython.display import display
import pandas as pd

if not result["update_screen"]:
    raise ValueError("缺失值审核仅适用于 screen 更新，请将 update_mode 设为 both 或 screen_only")

screen_path = paths["screen_path"]
latest_date = pd.Timestamp(result["month_date"])
all_dates = pd.DatetimeIndex(
    pd.to_datetime(pd.read_parquet(screen_path, columns=["Date"])["Date"].dropna().unique())
).sort_values()
history_dates = [date for date in all_dates if date < latest_date]

bench_map = {
    "全量公司": None,
    "MSCI WORLD": "Weight in MSCI WORLD",
    "SP500": "Weight in SP500",
    "STOXX EUROPE 600": "Weight in STOXX EUROPE 600",
}


def empty_stats(columns):
    return {
        bench_name: {
            "missing_count": pd.Series(0, index=columns, dtype="int64"),
            "row_count": 0,
        }
        for bench_name in bench_map
    }


# bench 口径统一按 Weight in XXX > 0 过滤

def update_stats(stats, df):
    for bench_name, weight_col in bench_map.items():
        bench_df = df if weight_col is None else df[df[weight_col].fillna(0) > 0]
        if bench_df.empty:
            continue
        stats[bench_name]["missing_count"] = (
            stats[bench_name]["missing_count"]
            .add(
                bench_df.isna().sum().reindex(df.columns, fill_value=0).astype("int64"),
                fill_value=0,
            )
            .astype("int64")
        )
        stats[bench_name]["row_count"] += len(bench_df)


latest_df = pd.read_parquet(screen_path, filters=[("Date", "==", latest_date)])
latest_stats = empty_stats(latest_df.columns)
history_stats = empty_stats(latest_df.columns)

update_stats(latest_stats, latest_df)

for history_date in history_dates:
    history_df = pd.read_parquet(screen_path, filters=[("Date", "==", history_date)])
    update_stats(history_stats, history_df)

missing_audit_results = {}
for bench_name in bench_map:
    latest_missing_count = latest_stats[bench_name]["missing_count"]
    latest_row_count = latest_stats[bench_name]["row_count"]
    history_missing_count = history_stats[bench_name]["missing_count"]
    history_row_count = history_stats[bench_name]["row_count"]

    audit_df = pd.DataFrame(
        {
            "latest_missing_count": latest_missing_count,
            "latest_row_count": latest_row_count,
            "latest_missing_rate": latest_missing_count / latest_row_count if latest_row_count else 0.0,
            "history_missing_count": history_missing_count,
            "history_row_count": history_row_count,
            "history_missing_rate": history_missing_count / history_row_count if history_row_count else 0.0,
        }
    )
    audit_df["delta_missing_rate"] = (
        audit_df["latest_missing_rate"] - audit_df["history_missing_rate"]
    )
    missing_audit_results[bench_name] = audit_df.sort_values(
        "delta_missing_rate",
        key=lambda s: s.abs(),
        ascending=False,
    )

for bench_name, audit_df in missing_audit_results.items():
    print(f"=== {bench_name} ===")
    display(audit_df)


In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display

from tp_data.monthly_update import build_default_paths
from tp_data.screen_func import ScreenProcessor

# 一键重算并回写全历史 Univ ML 权重
if "paths" not in globals():
    paths = build_default_paths()

screen_path = Path(paths["screen_path"])
last_screen_path = Path(paths["last_screen_path"])
processor = ScreenProcessor(str(paths["mapping_path"]), str(paths["returns_path"]))

history_df = pd.read_parquet(screen_path)
if "ISIN" not in history_df.columns:
    history_df = history_df.reset_index()

history_df["Date"] = pd.to_datetime(history_df["Date"])
if "Symbol" in history_df.columns:
    history_df["Symbol"] = history_df["Symbol"].astype("str")

# 先备份，再回写
backup_path = (
    screen_path.parent
    / "backups"
    / screen_path.stem
    / f"{screen_path.stem}_before_rewrite_univ_ml_{pd.Timestamp.now():%Y%m%d_%H%M%S}{screen_path.suffix}"
)
backup_path.parent.mkdir(parents=True, exist_ok=True)
history_df.set_index("ISIN").to_parquet(backup_path)

history_df = processor.add_univ_ml(history_df)
history_df = history_df.set_index("ISIN")
processor.validate_unique_keys(history_df)
processor.save_results(history_df, str(screen_path))

# 同步最新月到 last_screen
latest_date = history_df["Date"].max()
last_df = history_df.reset_index()
last_df = last_df[last_df["Date"] == latest_date].copy()
if "Symbol" in last_df.columns:
    last_df["Symbol"] = last_df["Symbol"].astype("str")
last_df = last_df.set_index("ISIN")
last_df.to_parquet(last_screen_path)

check_cols = ["Weight in Univ ML US", "Weight in Univ ML EU", "Weight in Univ ML OTHER"]
validation = history_df.reset_index().groupby("Date")[check_cols].sum().round(10)

print(f"历史已回写: {screen_path}")
print(f"最新快照已刷新: {last_screen_path}")
print(f"备份文件: {backup_path}")
display(validation.tail(12))


In [ ]:
# CIQ 合并已经纳入 run_monthly_update()。
# 这个单元只检查当前 input_month 批次中的 CIQ 输入文件，不再手工写 screen_aggregate。
from pathlib import Path

if "paths" not in globals():
    paths = build_default_paths()

ciq_input_dir = Path(paths["incoming_dir"]) / input_month / "ciq"
if not ciq_input_dir.is_dir():
    raise FileNotFoundError(f"CIQ 输入目录不存在: {ciq_input_dir}")

ciq_files = sorted(path for path in ciq_input_dir.iterdir() if path.is_file())
if not ciq_files:
    raise FileNotFoundError(f"CIQ 输入目录为空: {ciq_input_dir}")

ciq_files
